<a href="https://colab.research.google.com/github/vivomouallem12-oss/Cloud-Project/blob/main/GreenGuardCloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install firebase-admin

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving greenguardcloud-firebase-adminsdk-fbsvc-6cd265e46e.json to greenguardcloud-firebase-adminsdk-fbsvc-6cd265e46e (12).json


In [ ]:
list(uploaded.keys())

['greenguardcloud-firebase-adminsdk-fbsvc-6cd265e46e (12).json']

In [ ]:
# ==============================
# Connect to Firebase
# ==============================
import firebase_admin
from firebase_admin import credentials, firestore

SERVICE_ACCOUNT_FILE = "greenguardcloud-firebase-adminsdk-fbsvc-6cd265e46e.json"

cred = credentials.Certificate(SERVICE_ACCOUNT_FILE)

if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()

print("Connected to Firebase successfully!")

Connected to Firebase successfully!


In [ ]:
# ==============================
# Article Collection (5 academic papers on tomato diseases)
# ==============================
articles = {
    "article1": {
        "title": "Symptomology of major fungal diseases on tomato and its management",
        "authors": "S. Pavan Kumar, A. Srinivasulu, K. Raja Babu",
        "url": "https://www.researchgate.net/publication/383703523"
    },
    "article2": {
        "title": "Tomato Diseases: Identification, Biology and Control",
        "authors": "Henri Laterrot, Georges Marchoux, Thierry Candresse",
        "url": "https://doi.org/10.1201/b15145"
    },
    "article3": {
        "title": "Tomato diseases and disorders",
        "authors": "Mark L. Gleason, Brooke A. Edmunds",
        "url": "https://solverchembooks.com/msds/tomato-diseases-and-disorders-tomato-diseases-msds.pdf"
    },
    "article4": {
        "title": "A Review of the Most Common and Economically Important Diseases That Undermine the Cultivation of Tomato Crop in the Mediterranean Basin",
        "authors": "S. Panno, S. Davino, A. G. Caruso, S. Bertacca, A. Crnogorac, A. Mandić, E. Noris, S. Matić",
        "url": "https://www.mdpi.com/2073-4395/11/11/2188"
    },
    "article5": {
        "title": "The most important diseases caused by fungi in tomato seeds: A review",
        "authors": "Ali A. Alsudani",
        "url": "https://www.researchgate.net/publication/388026717"
    }
}

In [ ]:
# ==============================
# Inverted Index
# REQUIRED field names:
#   - term
#   - DocIDs
# ==============================

raw_mapping = {
    # General plant + disease
    "tomato":         ["article1", "article2", "article3", "article4", "article5"],
    "disease":        ["article1", "article2", "article3", "article4", "article5"],
    "pathogen":       ["article1", "article2", "article3", "article4", "article5"],
    "infection":      ["article1", "article2", "article3", "article4", "article5"],
    "symptom":        ["article1", "article2", "article3", "article4"],

    # Plant parts
    "leaf":           ["article1", "article2", "article3", "article4"],
    "root":           ["article1", "article2", "article3", "article5"],
    "stem":           ["article1", "article2", "article3", "article4"],
    "fruit":          ["article1", "article2", "article3", "article4"],
    "seed":           ["article2", "article3", "article5"],

    # Pathogen groups
    "fungi":          ["article1", "article2", "article3", "article4", "article5"],
    "bacteria":       ["article1", "article2", "article3", "article4"],
    "virus":          ["article1", "article2", "article3", "article4"],
    "nematode":       ["article1", "article2", "article4"],
    "spore":          ["article1", "article2", "article5"],

    # Symptoms
    "wilting":        ["article1", "article2", "article3", "article4"],
    "chlorosis":      ["article1", "article3", "article4"],
    "necrosis":       ["article1", "article3", "article4"],
    "spot":           ["article1", "article2", "article3", "article4"],
    "lesion":         ["article1", "article2", "article3", "article4"],
    "rot":            ["article1", "article2", "article3", "article4", "article5"],

    # Specific diseases
    "blight":         ["article1", "article2", "article3", "article4"],
    "mildew":         ["article1", "article3", "article4"],
    "mold":           ["article1", "article3", "article4", "article5"],
    "anthracnose":    ["article1", "article2", "article3", "article4"],
    "mosaic":         ["article2", "article4"],

    # Specific pathogens (genera)
    "fusarium":       ["article1", "article2", "article3", "article4", "article5"],
    "alternaria":     ["article1", "article2", "article3", "article4", "article5"],
    "septoria":       ["article2", "article4"],
    "phytophthora":   ["article1", "article2", "article3", "article4"],
    "verticillium":   ["article1", "article2", "article3", "article4"],
    "colletotrichum": ["article1", "article2", "article4", "article5"],
    "aspergillus":    ["article5"],
    "penicillium":    ["article5"],

    # Management / context
    "management":     ["article1", "article2", "article3", "article4", "article5"],
    "control":        ["article1", "article2", "article3", "article4"],
    "resistance":     ["article2", "article4"],
    "cultivation":    ["article2", "article4"],
    "yield":          ["article2", "article4", "article5"],
    "mediterranean":  ["article4"],
    "greenhouse":     ["article2", "article4"]
}

# Final index in the REQUIRED structure: list of {term, DocIDs}
inverted_index = [
    {"term": t, "DocIDs": docs} for t, docs in raw_mapping.items()
]

print(f"Total terms in index: {len(inverted_index)}")
print("\nFirst 5 entries:")
for entry in inverted_index[:5]:
    print(entry)

Total terms in index: 41

First 5 entries:
{'term': 'tomato', 'DocIDs': ['article1', 'article2', 'article3', 'article4', 'article5']}
{'term': 'disease', 'DocIDs': ['article1', 'article2', 'article3', 'article4', 'article5']}
{'term': 'pathogen', 'DocIDs': ['article1', 'article2', 'article3', 'article4', 'article5']}
{'term': 'infection', 'DocIDs': ['article1', 'article2', 'article3', 'article4', 'article5']}
{'term': 'symptom', 'DocIDs': ['article1', 'article2', 'article3', 'article4']}


In [ ]:
# ==============================
# Upload Articles to Firestore
# ==============================
for doc_id, data in articles.items():
    db.collection("articles").document(doc_id).set(data)

print("Articles uploaded successfully!")

Articles uploaded successfully!


In [ ]:
# ==============================
# Upload Index to Firestore
# ==============================
for term, doc_ids in raw_mapping.items():
    db.collection("index").document(term).set({
        "term": term,
        "DocIDs": doc_ids
    })

print("Index uploaded successfully!")

Index uploaded successfully!


In [ ]:
# ==============================
# Stop Words
# Filtered out as they carry no domain meaning
# ==============================
stop_words = [
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "with", "by", "from", "as",
    "and", "or", "but", "if", "then", "so", "that", "this", "these", "those",
    "it", "its", "we", "our", "they", "their", "have", "has", "had",
    "do", "does", "did", "can", "could", "will", "would", "should",
    "also", "such", "more", "most", "very", "much", "many", "some",
    "however", "therefore", "thus", "due"
]
# Rationale: these words appear in almost every English text and add no
# semantic value for plant-disease retrieval. Removing them sharpens the
# index toward domain-specific terms (disease names, plant parts, methods).

# ==============================
# Lemmatization / Stemming choice
# ==============================
# We applied LEMMATIZATION (not stemming) so that variants like
# "diseases" -> "disease", "leaves" -> "leaf", "spots" -> "spot"
# collapse to a single canonical term in the index.
# Lemmatization preserves real words (unlike stemming which can produce
# non-words like "diseas"), which improves the readability of the index
# and the quality of the RAG retrieval step.

print(f"✓ {len(stop_words)} stop words declared")

✓ 59 stop words declared


In [ ]:
# ==============================
# Lemmatization setup with NLTK WordNetLemmatizer
# ==============================
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_word(word: str) -> str:
    """Reduce a word to its base (lemma) form."""
    word = word.lower().strip()
    lemma = lemmatizer.lemmatize(word, pos='n')
    if lemma == word:
        lemma = lemmatizer.lemmatize(word, pos='v')
    return lemma

# Quick verification
test_words = ["diseases", "leaves", "spots", "lesions", "infecting", "pathogens"]
print("Lemmatization examples:")
for w in test_words:
    print(f"  {w}  ->  {lemmatize_word(w)}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Lemmatization examples:
  diseases  ->  disease
  leaves  ->  leaf
  spots  ->  spot
  lesions  ->  lesion
  infecting  ->  infect
  pathogens  ->  pathogen


In [ ]:
# ==============================
# Install Gemini SDK
# ==============================
!pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Option A (recommended): store the key in Colab Secrets
# Click the 🔑 icon on the left → add secret named "GEMINI_API_KEY"
try:
    GEMINI_API_KEY = userdata.get('Gemini_API_Key')
except Exception:
    # Option B: paste directly (less safe — don't commit to git!)
    GEMINI_API_KEY = ""

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('models/gemini-2.5-flash')
print("✓ Gemini ready")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ Gemini ready


In [ ]:
# ==============================
# Document snippets
# Copy each paper's abstract (or 2-3 key paragraphs) here.
# These are what the RAG will feed to the LLM.
# ==============================
snippets = {
    "article1": """
    Tomato (Lycopersicon esculentum) is the second most important vegetable crop
    next to potato. Tomato is a warm season crop, it requires warm and cool
    climate. The plant is highly affected by adverse climatic conditions. The
    warm and cool climatic conditions provide an ideal condition for the
    development of many foliar, stem and soil-borne plant diseases. Fungal
    diseases are a major limiting factor for vegetable that cause serious yield
    reduction leading to severe economic losses. The major fungal diseases in
    tomato are seedling Damping off, septoria leaf spot, Early blight, Late
    blight, Anthracnose, Powdery mildew, southern leaf blight, Fusarium wilt,
    Verticillium wilt, Buckeye rot. For each disease, main symptoms and their
    management practices mentioned. This review is based on combined
    information derived from available literature and expertise knowledge.
    Keywords: Powdery mildew, buckeye rot, economic losses, late blight.
    """,

    "article2": """
    [PASTE THE ABSTRACT of "Tomato Diseases: Identification, Biology and Control"
    here. If no abstract, paste the introduction's opening paragraphs.]
    """,
    "article3": """
    [PASTE THE ABSTRACT of "Tomato diseases and disorders" here.]
    """,

    "article4": """
    Tomato (Solanum lycopersicum L.), family Solanaceae, has become in the
    past fifty years one of the most important and extensively grown
    horticultural crops in the Mediterranean region and throughout the world.
    In 2019, more than 180 million tonnes of tomato have been produced
    worldwide, out of which around 42 million tonnes in Mediterranean
    countries. Due to its genetic properties, tomato is afflicted by numerous
    plant diseases induced by fungal, bacterial, phytoplasma, virus, and
    viroid pathogens. Not only is its genetic inheritance of great importance
    to the management of the numerous tomato pathogens, but equally as
    important are also the present climate changes, the recently revised
    phytopathological control measures, and the globalization of the seed
    industry. Thus, the recognition of symptoms and the knowledge of the
    distribution and spread of the disease and of the methods for early
    detection of the pathogens are the major prerequisites for a successful
    management of the disease. In this review, we describe the main tomato
    pathogens in the Mediterranean area that impact mostly the tomato yield
    and provide the current and perspective measures necessary for their
    successful management.

    Keywords: tomato diseases; fungi; bacteria; phytoplasmas; viruses;
    viroids; detection; crop management.

    Pathogens covered include Alternaria solani (early blight), Septoria
    lycopersici (leaf spot), Botrytis cinerea (grey mould), Fusarium oxysporum
    f. sp. lycopersici (Fusarium wilt), Fusarium oxysporum f. sp.
    radicis-lycopersici (crown and root rot), Verticillium dahliae
    (Verticillium wilt), Clavibacter michiganensis (bacterial canker),
    Pseudomonas syringae pv. tomato (bacterial speck), Candidatus Phytoplasma
    solani (stolbur disease), Tomato spotted wilt virus, Cucumber mosaic
    virus, Tomato yellow leaf curl virus, Tomato brown rugose fruit virus,
    Tomato mosaic virus, Parietaria mottle virus, Pepino mosaic virus, and
    Potato spindle tuber viroid.
    """,

    "article5": """
    Worldwide agriculture productivity and the economy are negatively impacted
    by plant diseases carried on by several microorganisms that are either
    present in the soil, seeds, or propagative planting materials, or that
    are conveyed by the air or water. This review provides a wealth of
    information about the importance and frequency of different types of
    seed-borne mycoflora such as Fusarium spp., Alternaria spp., Rhizopus
    spp., Aspergillus spp., Penicillium spp., Colletotrichum spp., which is
    specifically linked to tomato seeds.

    These mycoflora cause devastating tomato diseases such as grey mold,
    fruit and root rots, Fusarium wilt, early blight, and foot rots. The
    review evaluates a variety of contemporary and traditional methods that
    are used to identify seed-borne fungi and to implement various control
    measures, such as chemical and biological approaches. Many variables,
    such as the presence of vulnerable plants, ideal environmental
    circumstances, and overhead watering, pose significant barriers to the
    spread of plant diseases. In these circumstances, an efficient disease
    management strategy consists of monitoring plant health and detecting
    diseases, especially by screening infested seed lots before planting with
    the use of seed detection assays.

    The health aspects of seeds are also important to prevent the
    transmission of diseases to field crops. A wide variety of fungal
    pathogens are described, but the common among them are Fusarium spp.,
    Alternaria spp., Rhizopus spp., Aspergillus spp., Penicillium spp., and
    Rhizoctonia spp. Different species of Colletotrichum have also been
    reported as causing significant quality and quantity losses in tomato
    seeds.
    """
}

print(f"✓ {len(snippets)} snippets loaded")

✓ 5 snippets loaded


In [ ]:
# ==============================
# Multi-word retrieval
# Splits the user's query, lemmatizes each word,
# and finds documents containing ANY of them.
# Ranked by how many query terms each doc matches.
# ==============================
import re
from collections import Counter

def retrieve_documents(query, top_k=3):
    """Return the top_k most relevant article IDs for a query."""
    words = re.findall(r'\b[a-zA-Z]+\b', query.lower())
    query_terms = [lemmatize_word(w) for w in words if w not in stop_words]

    doc_scores = Counter()
    for term in query_terms:
        for entry in inverted_index:
            if entry["term"] == term:
                for doc_id in entry["DocIDs"]:
                    doc_scores[doc_id] += 1

    ranked = doc_scores.most_common(top_k)
    return [doc_id for doc_id, score in ranked]

# Quick test
print("Test 1:", retrieve_documents("What fungal diseases affect tomato leaves?"))
print("Test 2:", retrieve_documents("Fusarium wilt management"))
print("Test 3:", retrieve_documents("seed-borne pathogens"))

Test 1: ['article1', 'article2', 'article3']
Test 2: ['article1', 'article2', 'article3']
Test 3: ['article2', 'article3', 'article5']


In [ ]:
# ==============================
# RAG: Retrieval-Augmented Generation
# ==============================
def rag_answer(query, top_k=3):
    # Step 1: Retrieval
    relevant_ids = retrieve_documents(query, top_k=top_k)

    if not relevant_ids:
        return {"answer": "No relevant articles found.", "sources": []}

    # Step 2: Build context from snippets
    context = ""
    for doc_id in relevant_ids:
        meta = articles[doc_id]
        context += f"\n[{doc_id}] {meta['title']} ({meta['authors']})\n"
        context += f"{snippets[doc_id].strip()}\n"

    # Step 3: Build the prompt for Gemini
    prompt = f"""You are a helpful assistant for plant disease research.
Answer the user's question using ONLY the information provided in the articles below.
Cite the article ID in brackets after each claim, e.g. [article1].
If the articles don't contain enough information, say so honestly.

ARTICLES:
{context}

USER QUESTION: {query}

ANSWER (with citations):"""

    # Step 4: Generation - call Gemini
    response = model.generate_content(prompt)

    return {
        "answer": response.text,
        "sources": [
            {"id": doc_id,
             "title": articles[doc_id]["title"],
             "url": articles[doc_id]["url"]}
            for doc_id in relevant_ids
        ]
    }

In [ ]:
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get('Gemini_API_Key')
genai.configure(api_key=GOOGLE_API_KEY)

print("Models that support generateContent:")

available_models = []

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)
        available_models.append(m.name)

print("\nTotal:", len(available_models))

Models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-prev

In [ ]:
def check_rag_ready():
    print("Checking RAG variables...")

    if "articles" not in globals():
        print("❌ articles not found")
    else:
        print("articles type:", type(articles))

    if "rag_answer" not in globals():
        print("❌ rag_answer not found")
    else:
        print("✅ rag_answer found")

check_rag_ready()

Checking RAG variables...
articles type: <class 'dict'>
✅ rag_answer found


In [ ]:
# ============================================================
# GreenGuardCloud - Full Real Version + Improved UI
# Includes:
# 1. Real AI image disease detection
# 2. Real IoT sensor data
# 3. Real RAG connected to Firestore + Gemini
# 4. Real dashboard from latest IoT data
# 5. Extra feature: Weather + IoT smart care advice
# ============================================================

!pip install gradio pandas matplotlib pillow requests transformers torch -q

import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import requests
from transformers import pipeline


# ============================================================
# 1. REAL IoT DATA FROM SERVER
# ============================================================

BASE_URL = "https://server-cloud-v645.onrender.com/"

SENSOR_UNITS = {
    "temperature": "°C",
    "humidity": "%",
    "soil": "%",
    "json": "mixed data"
}

SENSOR_NAMES = {
    "temperature": "Temperature",
    "humidity": "Air Humidity",
    "soil": "Soil Moisture",
    "json": "Combined Sensor Data"
}


def fetch_iot_feed(feed="humidity", limit=10):
    try:
        limit = int(limit)

        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": limit},
            timeout=45
        )

        data = response.json()

        if "data" not in data:
            return pd.DataFrame([{"Error": str(data)}])

        df = pd.DataFrame(data["data"])

        if df.empty:
            return pd.DataFrame([{"Message": "No data found for this sensor."}])

        df["feed"] = feed

        if "created_at" in df.columns:
            df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

        if "value" in df.columns:
            df["value"] = pd.to_numeric(df["value"], errors="coerce")

        return df

    except Exception as e:
        return pd.DataFrame([{"Error": str(e)}])


def get_latest_value(feed):
    df = fetch_iot_feed(feed, limit=1)

    if df.empty:
        return None

    if "value" not in df.columns:
        return None

    value = df["value"].iloc[0]

    if pd.isna(value):
        return None

    return float(value)


def generate_iot_data(feed, limit):
    df = fetch_iot_feed(feed, limit)

    if df.empty:
        return df

    if "Error" in df.columns or "Message" in df.columns:
        return df

    rename_map = {
        "created_at": "Measurement Time",
        "value": "Sensor Value",
        "feed": "Sensor Type"
    }

    df = df.rename(columns=rename_map)

    df["Sensor Name"] = SENSOR_NAMES.get(feed, feed)
    df["Unit"] = SENSOR_UNITS.get(feed, "")

    wanted_columns = [
        "Measurement Time",
        "Sensor Name",
        "Sensor Value",
        "Unit",
        "Sensor Type"
    ]

    existing_columns = [col for col in wanted_columns if col in df.columns]

    return df[existing_columns]


# ============================================================
# 2. PLANT CONDITION ANALYSIS FROM REAL IoT DATA
# ============================================================

def analyze_real_plant_status(temp, humidity, soil):
    problems = []
    recommendations = []

    if temp is None:
        problems.append("Temperature data is missing.")
        recommendations.append("Check if the temperature sensor is connected and sending data.")
    elif temp < 20:
        problems.append("Temperature is too low for the plant.")
        recommendations.append("Move the plant to a warmer place or check the greenhouse temperature.")
    elif temp > 30:
        problems.append("Temperature is too high and may cause plant stress.")
        recommendations.append("Provide shade, improve ventilation, or water the plant carefully.")
    else:
        recommendations.append("Temperature is in a suitable range.")

    if humidity is None:
        problems.append("Air humidity data is missing.")
        recommendations.append("Check if the humidity sensor is connected and sending data.")
    elif humidity < 40:
        problems.append("Air humidity is too low.")
        recommendations.append("Increase air humidity or place water near the plant area.")
    elif humidity > 85:
        problems.append("Air humidity is too high and may encourage fungal diseases.")
        recommendations.append("Improve ventilation and avoid keeping leaves wet for a long time.")
    else:
        recommendations.append("Air humidity is in a suitable range.")

    if soil is None:
        problems.append("Soil moisture data is missing.")
        recommendations.append("Check if the soil moisture sensor is connected and sending data.")
    elif soil < 30:
        problems.append("Soil moisture is low. The plant may need watering.")
        recommendations.append("Water the plant and check the soil moisture again after a short time.")
    elif soil > 70:
        problems.append("Soil moisture is too high. There may be overwatering.")
        recommendations.append("Reduce watering and make sure the soil has good drainage.")
    else:
        recommendations.append("Soil moisture is in a suitable range.")

    if len(problems) == 0:
        text = "Plant condition is good ✅\n\n"
        text += "No serious problem was detected from the latest sensor readings.\n\n"
        text += "Recommendations:\n"
        text += "\n".join(["• " + r for r in recommendations])
        return text

    text = "Possible problems found:\n"
    text += "\n".join(["• " + p for p in problems])

    text += "\n\nRecommendations:\n"
    text += "\n".join(["• " + r for r in recommendations])

    return text


# ============================================================
# 3. REAL IMAGE UPLOAD + AI PLANT DISEASE DETECTION
# ============================================================

plant_image_model = None

def load_plant_image_model():
    global plant_image_model

    if plant_image_model is None:
        plant_image_model = pipeline(
            "image-classification",
            model="surgeonwz/plant-village"
        )

    return plant_image_model


def clean_prediction_label(label):
    label = str(label)
    label = label.replace("___", " - ")
    label = label.replace("_", " ")
    return label


def upload_plant_image(image):
    if image is None:
        return "### No image was uploaded.\n\nPlease upload a clear image of a plant leaf."

    try:
        model = load_plant_image_model()
        predictions = model(image)

        md = "### 🌿 Plant Image AI Analysis\n\n"
        md += "The uploaded image was analyzed using a real AI plant-disease classification model.\n\n"

        md += "### What the system does\n\n"
        md += (
            "The AI model checks the visual features of the leaf, such as spots, color changes, "
            "and damaged areas, then returns the most likely plant disease classes.\n\n"
        )

        md += "### Top Predictions\n\n"

        for pred in predictions[:5]:
            label = clean_prediction_label(pred["label"])
            score = pred["score"] * 100
            md += f"- **{label}** — `{score:.2f}%` confidence\n"

        md += "\n### Note\n\n"
        md += (
            "This is an AI prediction for a school project. "
            "It can help identify possible plant diseases, but it is not a final professional diagnosis. "
            "If the confidence is low, the result should be treated as a suggestion."
        )

        return md

    except Exception as e:
        return f"""
### Error while analyzing image

{str(e)}

Try running the cell again. The first time may take longer because Colab downloads the model.
"""


# ============================================================
# 4. REAL RAG FUNCTION FOR GRADIO
# This uses your existing rag_answer(query)
# ============================================================

def rag_gradio_search(query):
    if not query or query.strip() == "":
        return "### Please enter a question."

    if "rag_answer" not in globals():
        return """
### RAG is not loaded yet

Please run your Firebase/RAG cells first, especially the cells that define:

- `retrieve_documents(query)`
- `rag_answer(query)`
- `articles`
- `snippets`
- `model`

Then run this Gradio cell again.
"""

    try:
        result = rag_answer(query)

        md = f"### 🔍 User Question\n> {query}\n\n"

        md += "### 💡 AI Answer Based on Retrieved Sources\n\n"
        md += f"{result['answer']}\n\n"

        md += "### 📚 Sources Used\n\n"

        if "sources" in result and result["sources"]:
            for src in result["sources"]:
                source_id = src.get("id", "No ID")
                title = src.get("title", "Untitled source")
                url = src.get("url", "")

                md += f"- **[{source_id}]** *{title}*\n"
                if url:
                    md += f"  🔗 {url}\n"
        else:
            md += "- No specific sources matched.\n"

        md += "\n### Explanation\n\n"
        md += (
            "This answer was generated using RAG. "
            "RAG means that the system first searches the Firestore article database, "
            "retrieves relevant sources, and then sends the question with those sources to Gemini."
        )

        return md

    except Exception as e:
        return f"""
### Error while running RAG

{str(e)}
"""


# ============================================================
# 5. REAL DASHBOARD FROM LATEST IoT DATA
# ============================================================

def create_dashboard():
    temp = get_latest_value("temperature")
    humidity = get_latest_value("humidity")
    soil = get_latest_value("soil")

    df = pd.DataFrame({
        "Sensor": ["Temperature", "Air Humidity", "Soil Moisture"],
        "Value": [temp, humidity, soil],
        "Unit": ["°C", "%", "%"],
        "Meaning": [
            "Air temperature around the plant",
            "Humidity level in the air",
            "Moisture level inside the soil"
        ]
    })

    status = analyze_real_plant_status(temp, humidity, soil)

    plot_values = []
    for value in [temp, humidity, soil]:
        if value is None:
            plot_values.append(0)
        else:
            plot_values.append(value)

    fig = plt.figure(figsize=(8, 5))
    plt.bar(["Temperature °C", "Air Humidity %", "Soil Moisture %"], plot_values)
    plt.title("Real IoT Plant Dashboard")
    plt.ylabel("Latest Sensor Value")
    plt.xlabel("Sensor Type")
    plt.xticks(rotation=15)
    plt.tight_layout()

    return df, status, fig


# ============================================================
# 6. EXTRA FEATURE: WEATHER + IoT SMART CARE ADVICE
# Uses real weather API + real IoT sensor values
# ============================================================

def get_city_coordinates(city):
    try:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_response = requests.get(
            geo_url,
            params={
                "name": city,
                "count": 1,
                "language": "en",
                "format": "json"
            },
            timeout=30
        )

        geo_data = geo_response.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return None, None, None

        place = geo_data["results"][0]

        latitude = place["latitude"]
        longitude = place["longitude"]
        display_name = f"{place.get('name', city)}, {place.get('country', '')}"

        return latitude, longitude, display_name

    except Exception:
        return None, None, None


def get_weather_forecast(city):
    latitude, longitude, display_name = get_city_coordinates(city)

    if latitude is None or longitude is None:
        return None, f"Could not find weather data for: {city}"

    try:
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "forecast_days": 1,
                "timezone": "auto"
            },
            timeout=30
        )

        weather_data = weather_response.json()
        daily = weather_data["daily"]

        result = {
            "place": display_name,
            "date": daily["time"][0],
            "max_temp": daily["temperature_2m_max"][0],
            "min_temp": daily["temperature_2m_min"][0],
            "rain": daily["precipitation_sum"][0]
        }

        return result, None

    except Exception as e:
        return None, str(e)


def smart_weather_iot_advice(city):
    if not city or city.strip() == "":
        empty_df = pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        return empty_df, "### Please enter a city name."

    forecast, error = get_weather_forecast(city)

    if error:
        empty_df = pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        return empty_df, f"### Error\n\n{error}"

    # Real IoT values from the plant monitoring server
    iot_temp = get_latest_value("temperature")
    iot_humidity = get_latest_value("humidity")
    iot_soil = get_latest_value("soil")

    # Real weather values from Open-Meteo
    max_temp = forecast["max_temp"]
    min_temp = forecast["min_temp"]
    rain = forecast["rain"]

    summary_df = pd.DataFrame({
        "Source": [
            "Weather API",
            "Weather API",
            "Weather API",
            "IoT Sensor",
            "IoT Sensor",
            "IoT Sensor"
        ],
        "Metric": [
            "Maximum outside temperature",
            "Minimum outside temperature",
            "Expected rain",
            "Plant area temperature",
            "Air humidity near plant",
            "Soil moisture"
        ],
        "Value": [
            max_temp,
            min_temp,
            rain,
            iot_temp,
            iot_humidity,
            iot_soil
        ],
        "Unit": [
            "°C",
            "°C",
            "mm",
            "°C",
            "%",
            "%"
        ]
    })

    alerts = []
    recommendations = []

    # Combined logic: weather + plant sensors
    if rain > 5 and iot_soil is not None and iot_soil > 70:
        alerts.append("Rain is expected and soil moisture is already high.")
        recommendations.append("Do not water the plant today. Check drainage to avoid overwatering.")

    elif rain > 5 and iot_soil is not None and iot_soil < 30:
        alerts.append("Rain is expected, but current soil moisture is low.")
        recommendations.append("Wait for the rain first, then check the soil before watering.")

    elif rain == 0 and iot_soil is not None and iot_soil < 30:
        alerts.append("No rain is expected and soil moisture is low.")
        recommendations.append("Water the plant today and monitor the soil moisture after watering.")

    elif rain == 0 and iot_soil is not None and iot_soil >= 30:
        recommendations.append("No rain is expected, but soil moisture is not low. Water only if the soil becomes dry.")

    if max_temp >= 30 and iot_temp is not None and iot_temp >= 30:
        alerts.append("Both outside weather and plant-area temperature are high.")
        recommendations.append("Move the plant to shade, improve ventilation, and avoid watering during the hottest hours.")

    elif max_temp >= 30:
        alerts.append("High outside temperature is expected.")
        recommendations.append("Check the plant more often and provide shade if leaves look stressed.")

    if iot_humidity is not None and iot_humidity > 85 and rain > 0:
        alerts.append("Humidity is high and rain is expected.")
        recommendations.append("There is a higher risk of fungal disease. Improve airflow and avoid wet leaves.")

    elif iot_humidity is not None and iot_humidity < 40 and rain == 0:
        alerts.append("Air humidity is low and no rain is expected.")
        recommendations.append("Increase humidity around the plant and check watering needs.")

    if iot_temp is None or iot_humidity is None or iot_soil is None:
        alerts.append("Some IoT sensor values are missing.")
        recommendations.append("Check the IoT sensors or server connection.")

    if len(alerts) == 0:
        alerts.append("No serious combined weather and IoT problem was detected.")

    if len(recommendations) == 0:
        recommendations.append("Continue normal plant monitoring.")

    md = "### 🌦️ Smart Weather + IoT Plant Care Advice\n\n"
    md += f"**Location:** {forecast['place']}\n\n"
    md += f"**Date:** {forecast['date']}\n\n"

    md += "### Why this feature is stronger\n\n"
    md += (
        "This feature does not use only weather data. "
        "It combines real weather forecast data with the latest real IoT sensor readings "
        "from the plant monitoring system. Then it gives plant-care advice based on both sources.\n\n"
    )

    md += "### Alerts\n\n"
    for alert in alerts:
        md += f"- {alert}\n"

    md += "\n### Recommendations\n\n"
    for rec in recommendations:
        md += f"- {rec}\n"

    md += "\n### Data sources\n\n"
    md += "- Weather forecast: Open-Meteo API\n"
    md += "- Plant condition: IoT sensor server\n"

    return summary_df, md


# ============================================================
# 7. CLOSE OLD GRADIO APP IF EXISTS
# ============================================================

try:
    app.close()
except:
    pass


# ============================================================
# 8. GRADIO APP WITH 5 REAL SCREENS
# ============================================================

with gr.Blocks(title="GreenGuardCloud") as app:
    gr.Markdown("""
    # GreenGuardCloud 🌱

    Smart system for plant monitoring and plant disease support.

    This system includes five real parts:

    1. **Plant Image Upload** — uses a real AI model to predict possible plant diseases from an image.
    2. **IoT Sensor Data** — reads real sensor data from the plant monitoring server.
    3. **Search Engine** — searches Firestore sources and uses Gemini to answer questions.
    4. **Plant Dashboard** — shows the latest IoT readings, chart, alerts, and recommendations.
    5. **Smart Weather + IoT Advice** — combines real weather data with real IoT data to recommend plant care.
    """)

    # --------------------------------------------------------
    # Tab 1: Real AI image detection
    # --------------------------------------------------------
    with gr.Tab("1. Plant Image Upload"):
        gr.Markdown("""
        ### Upload a Plant Image for AI Disease Detection

        This screen allows the user to upload a clear image of a plant leaf.

        The image is analyzed by a real AI image-classification model.
        The system returns possible plant diseases and confidence scores.

        **How to use:**
        1. Upload a clear leaf image.
        2. Press **Analyze Image**.
        3. Read the top predictions and confidence percentages.
        """)

        plant_image = gr.Image(
            type="pil",
            label="Upload plant image"
        )

        image_result = gr.Markdown(label="AI Image Analysis")
        image_button = gr.Button("Analyze Image")

        image_button.click(
            fn=upload_plant_image,
            inputs=plant_image,
            outputs=image_result
        )

    # --------------------------------------------------------
    # Tab 2: Real IoT data
    # --------------------------------------------------------
    with gr.Tab("2. IoT Sensor Data"):
        gr.Markdown("""
        ### Real IoT Sensor Data

        This screen shows real sensor readings from the plant monitoring server.

        **Sensor types:**
        - `temperature` = air temperature in °C
        - `humidity` = air humidity in %
        - `soil` = soil moisture in %
        - `json` = combined or raw sensor data

        **How to use:**
        1. Choose a sensor type.
        2. Choose the number of samples.
        3. Press **Get IoT Data**.
        4. The table will show measurement time, value, unit, and sensor type.
        """)

        feed_dropdown = gr.Dropdown(
            choices=["humidity", "soil", "temperature", "json"],
            value="humidity",
            label="Choose sensor type"
        )

        limit_slider = gr.Slider(
            minimum=1,
            maximum=100,
            value=10,
            step=1,
            label="Number of samples"
        )

        iot_output = gr.Dataframe(
            label="Sensor Data Table",
            headers=["Measurement Time", "Sensor Name", "Sensor Value", "Unit", "Sensor Type"],
            value=pd.DataFrame(columns=["Measurement Time", "Sensor Name", "Sensor Value", "Unit", "Sensor Type"])
        )

        iot_button = gr.Button("Get IoT Data")

        iot_button.click(
            fn=generate_iot_data,
            inputs=[feed_dropdown, limit_slider],
            outputs=iot_output
        )

    # --------------------------------------------------------
    # Tab 3: Real RAG
    # --------------------------------------------------------
    with gr.Tab("3. Search Engine"):
        gr.Markdown("""
        ### Ask a Question About Plant Diseases

        This screen is a smart search engine.

        It uses **RAG**, which means:
        1. The system searches the Firestore article database.
        2. It retrieves relevant plant disease sources.
        3. It sends the question and the sources to Gemini.
        4. Gemini generates an answer based on the retrieved sources.

        **How to use:**
        Type a question about plant diseases, then press **Search**.
        """)

        query = gr.Textbox(
            label="Enter your plant disease question",
            placeholder="Example: why are tomato leaves yellow?"
        )

        search_result = gr.Markdown(label="RAG Answer")
        search_button = gr.Button("Search")

        search_button.click(
            fn=rag_gradio_search,
            inputs=query,
            outputs=search_result
        )

    # --------------------------------------------------------
    # Tab 4: Real IoT dashboard
    # --------------------------------------------------------
    with gr.Tab("4. Plant Dashboard"):
        gr.Markdown("""
        ### Real Plant Condition Dashboard

        This dashboard uses the latest real IoT readings:
        temperature, air humidity, and soil moisture.

        The dashboard shows:
        - Latest sensor values
        - Measurement units
        - A visual chart
        - Plant status alerts
        - Care recommendations

        **How to use:**
        Press **Create Dashboard** to load the latest sensor data and analyze plant condition.
        """)

        dashboard_table = gr.Dataframe(
            label="Latest Sensor Data",
            headers=["Sensor", "Value", "Unit", "Meaning"],
            value=pd.DataFrame(columns=["Sensor", "Value", "Unit", "Meaning"])
        )

        dashboard_status = gr.Textbox(
            label="Plant Status Alerts and Recommendations",
            lines=10
        )

        dashboard_plot = gr.Plot(label="Dashboard Chart")
        dashboard_button = gr.Button("Create Dashboard")

        dashboard_button.click(
            fn=create_dashboard,
            inputs=None,
            outputs=[
                dashboard_table,
                dashboard_status,
                dashboard_plot
            ]
        )

    # --------------------------------------------------------
    # Tab 5: Extra feature - Weather + IoT advice
    # --------------------------------------------------------
    with gr.Tab("5. Smart Weather + IoT Advice"):
        gr.Markdown("""
        ### Smart Weather + IoT Plant Care Advice

        This is the extra feature of the system.

        It combines:
        - Real weather forecast from an external weather API
        - Real plant sensor values from the IoT server

        The system then gives practical plant-care recommendations.

        **How to use:**
        1. Enter a city name.
        2. Press **Get Smart Advice**.
        3. Read the table, alerts, and recommendations.
        """)

        city_input = gr.Textbox(
            label="Enter city name",
            placeholder="Example: Beer Sheva"
        )

        smart_table = gr.Dataframe(
            label="Weather + IoT Data Used for Recommendation",
            headers=["Source", "Metric", "Value", "Unit"],
            value=pd.DataFrame(columns=["Source", "Metric", "Value", "Unit"])
        )

        smart_output = gr.Markdown(label="Smart Advice")
        smart_button = gr.Button("Get Smart Advice")

        smart_button.click(
            fn=smart_weather_iot_advice,
            inputs=city_input,
            outputs=[smart_table, smart_output]
        )


app.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e6305ad072e43c5e48.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e6305ad072e43c5e48.gradio.live
